# Real vs Synthetic Training Comparison
Same frozen wav2vec2-base-960h feature extractor, same LogisticRegression classifier.
Only the **training data** differs:

1. **Real Baseline** — trained on real lvPPA (89 dysfluent) + segmentedcc (235 control)
2. **Synthetic** — trained on concatenated synthetic data, moderate+severe only (15s window sampling)

**Test set**: JHU (74 dysfluent) + Capilouto (311 control) — completely held-out groups.

In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import glob
import random
import numpy as np
import torch
import torchaudio
import torch.nn.functional as F
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SR = 16000
WINDOW_SEC = 15
WINDOW_SAMPLES = WINDOW_SEC * TARGET_SR  # 240,000
SAMPLES_PER_STREAM = 3

print(f"Device: {DEVICE}")

Device: cuda


In [12]:
# ── Collect data ──
REAL_DIR = os.path.join(BASE_DIR, "data", "real")

# REAL training data: lvPPA (dysfluent) + segmentedcc (control)
real_dys_files = sorted(glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True))
real_ctrl_files = sorted(glob.glob(os.path.join(REAL_DIR, "*segmentedcc*", "**", "*.wav"), recursive=True))
real_files = real_dys_files + real_ctrl_files
real_labels = np.array([1] * len(real_dys_files) + [0] * len(real_ctrl_files))

# Train/val split for real data (80/20)
real_train_idx, real_val_idx = train_test_split(
    np.arange(len(real_files)), test_size=0.2, random_state=SEED, stratify=real_labels
)

# TEST data: JHU (dysfluent) + Capilouto (control) — completely held out
test_dys_files = sorted(glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True))
test_ctrl_files = sorted(glob.glob(os.path.join(REAL_DIR, "*Capilouto*", "**", "*.wav"), recursive=True))
test_files = test_dys_files + test_ctrl_files
test_labels = np.array([1] * len(test_dys_files) + [0] * len(test_ctrl_files))

# SYNTHETIC training data: concat streams (moderate + severe only, no mild)
CONCAT_DIR = os.path.join(BASE_DIR, "data")
synth_ctrl_files = sorted(glob.glob(os.path.join(CONCAT_DIR, "concat_control", "*.wav")))
synth_dys_files = sorted(
    glob.glob(os.path.join(CONCAT_DIR, "concat_dysfluent", "moderate", "*.wav"))
    + glob.glob(os.path.join(CONCAT_DIR, "concat_dysfluent", "severe", "*.wav"))
)
synth_files = synth_ctrl_files + synth_dys_files
synth_labels = np.array([0] * len(synth_ctrl_files) + [1] * len(synth_dys_files))

print("=== Real Training Data (lvPPA + segmentedcc) ===")
print(f"  Dysfluent (lvPPA):  {len(real_dys_files)}")
print(f"  Control (segcc):    {len(real_ctrl_files)}")
print(f"  Train: {len(real_train_idx)}  Val: {len(real_val_idx)}")

print(f"\n=== Synthetic Training Data (concat streams) ===")
print(f"  Control streams:    {len(synth_ctrl_files)}")
print(f"  Dysfluent streams:  {len(synth_dys_files)} (moderate+severe)")
print(f"  Total streams:      {len(synth_files)}")
print(f"  Samples per stream: {SAMPLES_PER_STREAM} x {WINDOW_SEC}s windows")

print(f"\n=== Test Data (JHU + Capilouto) ===")
print(f"  Dysfluent (JHU):    {len(test_dys_files)}")
print(f"  Control (Capilouto):{len(test_ctrl_files)}")
print(f"  Total:              {len(test_files)}")

=== Real Training Data (lvPPA + segmentedcc) ===
  Dysfluent (lvPPA):  89
  Control (segcc):    235
  Train: 259  Val: 65

=== Synthetic Training Data (concat streams) ===
  Control streams:    220
  Dysfluent streams:  220 (moderate+severe)
  Total streams:      440
  Samples per stream: 3 x 15s windows

=== Test Data (JHU + Capilouto) ===
  Dysfluent (JHU):    74
  Control (Capilouto):311
  Total:              385


In [13]:
# ── Load frozen wav2vec2 + feature extraction helpers ──
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
model.config.mask_time_prob = 0.0
model.config.mask_feature_prob = 0.0
model.eval().to(DEVICE)

def mean_pool(hidden, attn_mask):
    """Mean-pool hidden states accounting for CNN downsampling."""
    input_lengths = attn_mask.sum(dim=1)
    output_lengths = model._get_feat_extract_output_lengths(input_lengths).long().clamp(min=1)
    out_mask = torch.arange(hidden.size(1), device=hidden.device).unsqueeze(0) < output_lengths.unsqueeze(1)
    pooled = (hidden * out_mask.unsqueeze(-1)).sum(1) / output_lengths.unsqueeze(1).float()
    return pooled

def extract_features_from_files(files):
    """Extract mean-pooled 768-dim features for a list of WAV files (variable length)."""
    features = []
    for f in tqdm(files, leave=False):
        wav, sr = torchaudio.load(f)
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR).mean(0)
        inputs = processor(wav, sampling_rate=TARGET_SR, return_tensors="pt")
        input_values = inputs.input_values.to(DEVICE)
        attn_mask = torch.ones_like(input_values)
        with torch.no_grad():
            hidden = model(input_values, attention_mask=attn_mask).last_hidden_state
            pooled = mean_pool(hidden, attn_mask)
        features.append(pooled.cpu().numpy().squeeze())
    return np.array(features)

def extract_features_from_streams(files, labels, samples_per_stream=SAMPLES_PER_STREAM):
    """Extract features from concat streams by sampling random 15s windows."""
    all_features = []
    all_labels = []
    for i, f in enumerate(tqdm(files, leave=False)):
        wav, sr = torchaudio.load(f)
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR).squeeze(0)
        for _ in range(samples_per_stream):
            if len(wav) <= WINDOW_SAMPLES:
                chunk = F.pad(wav, (0, WINDOW_SAMPLES - len(wav)))
            else:
                start = random.randint(0, len(wav) - WINDOW_SAMPLES)
                chunk = wav[start:start + WINDOW_SAMPLES]
            inputs = processor(chunk, sampling_rate=TARGET_SR, return_tensors="pt")
            input_values = inputs.input_values.to(DEVICE)
            attn_mask = torch.ones_like(input_values)
            with torch.no_grad():
                hidden = model(input_values, attention_mask=attn_mask).last_hidden_state
                pooled = mean_pool(hidden, attn_mask)
            all_features.append(pooled.cpu().numpy().squeeze())
            all_labels.append(labels[i])
    return np.array(all_features), np.array(all_labels)

print("Model loaded, feature extraction functions ready.")

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 565.36it/s, Materializing param=feature_projection.projection.weight]                         
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded, feature extraction functions ready.


In [14]:
# ── Extract features ──

# Real data features (lvPPA + segmentedcc)
print("Extracting real data features (lvPPA + segmentedcc)...")
X_real = extract_features_from_files(real_files)
print(f"  Real features: {X_real.shape}")

# Test data features (JHU + Capilouto)
print("Extracting test data features (JHU + Capilouto)...")
X_test = extract_features_from_files(test_files)
print(f"  Test features: {X_test.shape}")

# Synthetic data features (concat streams, 15s windows)
print(f"Extracting synthetic features ({len(synth_files)} streams x {SAMPLES_PER_STREAM} windows)...")
X_synth, y_synth = extract_features_from_streams(synth_files, synth_labels)
print(f"  Synthetic features: {X_synth.shape}")
print(f"  Synthetic labels: {y_synth.sum()} dysfluent, {len(y_synth) - y_synth.sum()} control")

Extracting real data features (lvPPA + segmentedcc)...


  Real features: (324, 768)
Extracting test data features (JHU + Capilouto)...


  Test features: (385, 768)
Extracting synthetic features (440 streams x 3 windows)...


  Synthetic features: (1320, 768)
  Synthetic labels: 660 dysfluent, 660 control


In [15]:
# ── Cross-validation on training data ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["f1_macro", "accuracy", "roc_auc"]

print("5-fold Stratified CV on training data:\n")
cv_results = {}

# Real baseline CV
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
scores = cross_validate(pipe, X_real, real_labels, cv=cv, scoring=scoring)
cv_results["Real"] = scores
print(
    f"{'Real':>12s}  "
    f"F1={scores['test_f1_macro'].mean():.3f}+/-{scores['test_f1_macro'].std():.3f}  "
    f"Acc={scores['test_accuracy'].mean():.3f}+/-{scores['test_accuracy'].std():.3f}  "
    f"AUC={scores['test_roc_auc'].mean():.3f}+/-{scores['test_roc_auc'].std():.3f}"
)

# Synthetic CV
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
scores = cross_validate(pipe, X_synth, y_synth, cv=cv, scoring=scoring)
cv_results["Synthetic"] = scores
print(
    f"{'Synthetic':>12s}  "
    f"F1={scores['test_f1_macro'].mean():.3f}+/-{scores['test_f1_macro'].std():.3f}  "
    f"Acc={scores['test_accuracy'].mean():.3f}+/-{scores['test_accuracy'].std():.3f}  "
    f"AUC={scores['test_roc_auc'].mean():.3f}+/-{scores['test_roc_auc'].std():.3f}"
)

5-fold Stratified CV on training data:

        Real  F1=0.908+/-0.039  Acc=0.926+/-0.032  AUC=0.976+/-0.008
   Synthetic  F1=0.973+/-0.011  Acc=0.973+/-0.011  AUC=0.995+/-0.004


In [16]:
# ── Test set evaluation (JHU + Capilouto) ──
print("Test Set Evaluation (JHU dysfluent + Capilouto control):\n")

test_results = {}
for name, X_train, y_train in [
    ("Real", X_real, real_labels),
    ("Synthetic", X_synth, y_synth),
]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]

    f1 = f1_score(test_labels, preds, average="macro")
    acc = (preds == test_labels).mean()
    auc = roc_auc_score(test_labels, probs)
    test_results[name] = {"f1": f1, "acc": acc, "auc": auc, "preds": preds, "probs": probs}

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  F1 Macro: {f1:.4f}   Accuracy: {acc:.4f}   AUC: {auc:.4f}")
    print(classification_report(test_labels, preds, target_names=["control", "dysfluent"]))

Test Set Evaluation (JHU dysfluent + Capilouto control):


  Real
  F1 Macro: 0.6231   Accuracy: 0.6701   AUC: 0.7914
              precision    recall  f1-score   support

     control       0.94      0.63      0.76       311
   dysfluent       0.35      0.82      0.49        74

    accuracy                           0.67       385
   macro avg       0.64      0.73      0.62       385
weighted avg       0.82      0.67      0.71       385


  Synthetic
  F1 Macro: 0.6788   Accuracy: 0.7195   AUC: 0.9107
              precision    recall  f1-score   support

     control       0.98      0.67      0.79       311
   dysfluent       0.40      0.95      0.56        74

    accuracy                           0.72       385
   macro avg       0.69      0.81      0.68       385
weighted avg       0.87      0.72      0.75       385



In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MISC_DIR = os.path.join(BASE_DIR, "data", "misc")
os.makedirs(MISC_DIR, exist_ok=True)

models = ["Real", "Synthetic"]
colors = ["tab:blue", "tab:orange"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (metric, label) in zip(axes, [("f1", "F1 Macro"), ("acc", "Accuracy"), ("auc", "AUC-ROC")]):
    vals = [test_results[m][metric] for m in models]
    bars = ax.bar(models, vals, color=colors, edgecolor="black", linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontsize=10)
    ax.set_ylabel(label)
    ax.set_ylim(0, 1.05)

fig.suptitle(
    f"Test on JHU ({len(test_dys_files)} dys) + Capilouto ({len(test_ctrl_files)} ctrl)\n"
    f"Real = lvPPA+segcc ({len(real_files)})  |  Synthetic = concat streams ({len(synth_files)} x {SAMPLES_PER_STREAM} = {len(y_synth)})",
    fontsize=10,
)
fig.tight_layout()
fig.savefig(os.path.join(MISC_DIR, "real_vs_synthetic.png"), dpi=150)
plt.close(fig)
print(f"Saved to {MISC_DIR}/real_vs_synthetic.png")

Saved to /data/liharrison/lvsim/data/misc/real_vs_synthetic.png
